# STRATA, sub-células por tensor no kernel: Llama-3.1-8B

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução* → **A100** com **RAM alta**. Depois *Executar tudo*.

A vertente 3 escolhida na seção 28 de `docs/05-resultados.md`, agora nos kernels: os tipos `STRATA_Q4_K_D1/D2` e `STRATA_Q6_K_D1/D2` usam os mesmos blocos dos tipos P e uma tabela de 16 bytes por tensor (chave `strata.tab.<tensor>` do GGUF) com a posição de cada filho de uma célula. O nível 0 continua o Q4_K_M com imatrix intacto, byte a byte.

O kernel usa a variante **borda**: o deslocamento depende só da classe do código do nível 0 (menor, maior ou meio) e dos bits dos níveis de baixo. A emulação da seção 28 usava um deslocamento por código (16 ou 64). No Llama-3.2-1B as duas deram a mesma KL; este notebook confere isso no 8B.

O que ele mede, na mesma sessão e com a mesma imatrix:
1. **Conferência do CUDA**: `strata_cuda_check` compara os tipos P e D na GPU com o CPU (produto escalar MMVQ e desquantização + cuBLAS).
2. **Caminho real**: KL e velocidade dos GGUFs P (o STRATA de hoje) e D (sub-células), níveis 1 e 2.
3. **Curva justa**: na GPU, os tipos P e D passam pela desquantização para FP16 + cuBLAS no prompt e os k-quants pelo MMQ, que arredonda as ativações para 8 bits. Por isso o STRATA é comparado com os k-quants desquantizados para F16 (o mesmo caminho), como na seção 28.
4. **Emulação** (F16) das sub-células por código e por borda, para ver se a variante do kernel perde algo e se o kernel dá a mesma KL que a emulação.

**Régua (a mesma da seção 28):** bits por peso a mais que o k-quant com imatrix de mesma KL, por nível, com o nível 1 contado com o Q6_K empacotado. Seção 28: hoje 0,34 / 0,20; sub-células por tensor (emulação) 0,21 / 0,10 (níveis 1 / 2). Velocidade: geração ≥ 0,95 da do tipo P.

Ocupa ~60 GB de disco no pico (cada GGUF F16 é apagado depois de medido).

In [ ]:
EMULAR = ['codigo', 'borda']   # emulação F16 das sub-células por tensor ([] = só o kernel)
KLD_CHUNKS = 100           # trechos na divergência KL (~65 MB de disco por trecho)
IMATRIX_CHUNKS = 125       # trechos de 512 tokens do wikitext de treino para a imatrix
MODELO_HF = "unsloth/Meta-Llama-3.1-8B"

In [ ]:
import os, re, json, shutil, subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'Sem GPU NVIDIA: troque o tipo de ambiente de execução para A100 (ou L4).'
GPU = gpu.stdout.strip().splitlines()[0]
N = os.cpu_count()
RAM = int(next(l for l in open('/proc/meminfo') if l.startswith('MemTotal')).split()[1]) / 1e6
DISCO = shutil.disk_usage('/content').free / 1e9
print(f'GPU: {GPU} | CPU: {N} threads | RAM {RAM:.0f} GB | disco livre {DISCO:.0f} GB')
if DISCO < 75:
    print('Aviso: pouco disco. Ponha KLD_CHUNKS menor ou EMULAR = [].')
if RAM < 50:
    print('Aviso: pouca RAM para a emulação. Ligue a RAM alta.')

In [ ]:
BRANCH = "claude/project-thread-qjksmf"
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"   # o commit em que o patch dos tipos foi feito
!rm -rf strata && git clone -q --depth 1 -b $BRANCH https://github.com/Cantyll/strata.git
!rm -rf llama.cpp && git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp.git && git fetch -q --depth 1 origin $LLAMA_SHA && git checkout -q FETCH_HEAD
!sh strata/kernel/ggml/aplicar.sh llama.cpp
!pip install -q -U transformers safetensors
!pip install -q -e llama.cpp/gguf-py
# o fim do log da compilação fica visível, para aparecer qualquer erro
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DGGML_NATIVE=ON 2>&1 | tail -3 && cmake --build build -j $(nproc) --target llama-quantize llama-perplexity llama-imatrix llama-bench 2>&1 | tail -15
!ls -la llama.cpp/build/bin | grep -E 'llama-(perplexity|quantize|imatrix|bench)$'

Conferência do CUDA: cada tipo P e D (tabela qualquer) multiplicado por ativações com 1 a 512 colunas, na GPU e no CPU. Tem que terminar em `tudo OK`.

In [ ]:
LB = '/content/llama.cpp'
!cc -O2 strata/tests/kernel/strata_cuda_check.c -I {LB}/ggml/include -L {LB}/build/bin -lggml -lggml-base -lggml-cpu -lggml-cuda -lm -Wl,-rpath,{LB}/build/bin -o strata_cuda_check
out = !./strata_cuda_check
print('\n'.join(l for l in out if 'DIVERGE' in l or 'N=1 ' in l or 'N=512' in l or l is out[-1]))
CUDA_OK = out[-1].strip() == 'tudo OK'
assert CUDA_OK, 'O CUDA diverge do CPU: cole a saída acima na thread.'

In [ ]:
import urllib.request, zipfile, io
try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip').read()))
    open('/content/wiki.test.raw', 'wb').write(z.read('wikitext-2-raw/wiki.test.raw'))
    open('/content/wiki.train.raw', 'wb').write(z.read('wikitext-2-raw/wiki.train.raw'))
    TEXTO = 'wikitext-2 raw'
except Exception as e:
    print('download do wikitext raw falhou:', e)
    base = 'https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/'
    urllib.request.urlretrieve(base + 'test.txt', '/content/wiki.test.raw')
    urllib.request.urlretrieve(base + 'train.txt', '/content/wiki.train.raw')
    TEXTO = 'wikitext-2 (não-raw, cópia do PyTorch)'
print(TEXTO, os.path.getsize('/content/wiki.test.raw'), 'bytes de teste,', os.path.getsize('/content/wiki.train.raw'), 'de treino')

Modelo em BF16, imatrix, k-quants com imatrix e a base da KL. Os pesos originais ficam até os GGUFs do STRATA serem gravados.

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download(MODELO_HF, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
G = '/content/gguf'
os.makedirs(G, exist_ok=True)
!python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile {G}/BF16.gguf 2>&1 | tail -1
!llama.cpp/build/bin/llama-imatrix -m {G}/BF16.gguf -f /content/wiki.train.raw -ngl 99 -c 512 --chunks {IMATRIX_CHUNKS} -o {G}/imatrix.gguf > {G}/imatrix.log 2>&1
!tail -2 {G}/imatrix.log
for q in ['Q3_K_M', 'Q4_K_M', 'Q5_K_M', 'Q6_K']:
    !llama.cpp/build/bin/llama-quantize --imatrix {G}/imatrix.gguf {G}/BF16.gguf {G}/{q}_imat.gguf {q} > {G}/{q}_imat.log 2>&1
!llama.cpp/build/bin/llama-quantize {G}/BF16.gguf {G}/Q8_0.gguf Q8_0 > {G}/Q8_0.log 2>&1
!llama.cpp/build/bin/llama-perplexity -m {G}/BF16.gguf -f /content/wiki.test.raw -c 512 -ngl 99 --chunks {KLD_CHUNKS} --kl-divergence-base /content/kld_base.bin > /dev/null 2>&1
!ls -la {G}/*.gguf /content/kld_base.bin
!df -h /content | tail -1

Funções de medida. `tam` guarda o tamanho de cada GGUF antes de ele ser apagado.

In [ ]:
kld, tam, vel, tabelas = {}, {}, {}, {}
def kl_de(path, nome):
    out = !llama.cpp/build/bin/llama-perplexity -m {path} --kl-divergence-base /content/kld_base.bin --kl-divergence -ngl 99 2>&1
    txt = '\n'.join(out)
    k_ = re.search(r'Mean +KLD: +([0-9.]+) ± +([0-9.]+)', txt)
    top = re.search(r'Same top p: +([0-9.]+) ± +([0-9.]+) %', txt)
    if not k_:
        print(f'Erro na KL de {nome}:', '\n'.join(out[-8:]))
        return
    kld[nome] = (float(k_.group(1)), float(k_.group(2)), float(top.group(1)))
    tam[nome] = os.path.getsize(path) / 2**30
    print(f'{nome}: KL ×1000 {1000 * kld[nome][0]:.2f}, mesmo 1º token {kld[nome][2]}%, {tam[nome]:.3f} GiB')

def bench(path, nome):
    # tok/s de geração (tg128) e de prompt (pp512) na GPU, 3 repetições
    out = !llama.cpp/build/bin/llama-bench -m {path} -ngl 99 -p 512 -n 128 -r 3 -o json 2>/dev/null
    try:
        r = json.loads('\n'.join(l for l in out if not l.startswith('ggml_') and not l.startswith('load')))
        vel[nome] = {('pp' if x['n_prompt'] else 'tg'): x['avg_ts'] for x in r}
        print(f'{nome}: {vel[nome]}')
    except Exception as e:
        print(f'Erro no llama-bench de {nome}:', e, '\n'.join(out[-5:]))

Caminho real: k-quants com imatrix (KL pelo MMQ, só de referência; velocidade).

In [ ]:
for q in ['Q3_K_M_imat', 'Q4_K_M_imat', 'Q5_K_M_imat', 'Q6_K_imat', 'Q8_0']:
    kl_de(f'{G}/{q}.gguf', q)
for q in ['Q5_K_M_imat', 'Q6_K_imat']:
    bench(f'{G}/{q}.gguf', q)

O STRATA no caminho real: tipos P (hoje) e D (sub-células por tensor), níveis 1 e 2 sobre o mesmo Q4_K_M com imatrix. A gravação dos D faz duas passadas por tensor (ajuste da tabela, depois os blocos).

In [ ]:
%cd /content/strata
!python tests/kernel/strata_gguf_niveis.py --model {src} --gguf {G}/Q4_K_M_imat.gguf --nivel 1 2 --out {G}/P_N1.gguf {G}/P_N2.gguf 2>&1 | grep -v Warning | tail -2
!python tests/kernel/strata_gguf_niveis.py --model {src} --gguf {G}/Q4_K_M_imat.gguf --nivel 1 2 --sub-celulas --imatrix {G}/imatrix.gguf --out {G}/D_N1.gguf {G}/D_N2.gguf 2>&1 | grep -v Warning | tail -2
%cd /content
!rm -rf {src}
for n in ['P_N1', 'P_N2', 'D_N1', 'D_N2']:
    kl_de(f'{G}/{n}.gguf', n)
    bench(f'{G}/{n}.gguf', n)
    !rm -f {G}/{n}.gguf

Caminho F16: k-quants desquantizados (a curva justa para os tipos P e D na GPU), o STRATA de hoje e as sub-células por tensor emuladas. Cada GGUF F16 tem ~16 GB.

In [ ]:
ESC = f'python strata/tests/kernel/strata_escala_nivel.py --bf16 {G}/BF16.gguf --imatrix {G}/imatrix.gguf'
for q in ['Q4_K_M_imat', 'Q5_K_M_imat', 'Q6_K_imat', 'Q8_0']:
    !{ESC} --gguf {G}/{q}.gguf --modo copia --out /content/e.gguf 2>&1 | grep -v Warning | tail -1
    kl_de('/content/e.gguf', 'F16_' + q)
    tam['F16_' + q] = tam[q]
    !rm -f /content/e.gguf
    if q != 'Q4_K_M_imat':
        !rm -f {G}/{q}.gguf
!{ESC} --gguf {G}/Q4_K_M_imat.gguf --modo fixo --out /content/e1.gguf /content/e2.gguf 2>&1 | grep -v Warning | grep -v '^ *vals'
kl_de('/content/e1.gguf', 'F16_fixo_N1')
kl_de('/content/e2.gguf', 'F16_fixo_N2')
!rm -f /content/e1.gguf /content/e2.gguf
for por in EMULAR:
    out = !{ESC} --gguf {G}/Q4_K_M_imat.gguf --modo desloc --grupo tensor --por {por} --out /content/e1.gguf /content/e2.gguf 2>&1
    out = [l for l in out if 'Warning' not in l]
    print('\n'.join(out[-3:]))
    for k in (1, 2):
        kl_de(f'/content/e{k}.gguf', f'F16_{por}_N{k}')
        m = re.search(rf'nível {k}, .*tabelas de ([0-9.]+) MiB', '\n'.join(out))
        tabelas[f'F16_{por}_N{k}'] = float(m.group(1)) if m else float('nan')
    !rm -f /content/e1.gguf /content/e2.gguf

**A régua.** Para cada nível:
- *RAM*: o tamanho do GGUF no caminho real; nas emulações, o do tipo P no mesmo nível mais as tabelas. Na coluna "Q6_K empacotado", o nível 1 sem o byte inteiro por peso nas matrizes Q6_K, que é o que um tipo empacotado daria.
- *bits a mais*: quantos bits por peso o candidato gasta além do k-quant com imatrix de mesma KL, pela curva em F16 (o mesmo caminho de ativações dos tipos P e D na GPU). Abaixo de 0 = o STRATA ganha.
- *razão*: KL do candidato dividida pela do k-quant em F16 no mesmo tamanho.
- *D / emulação*: o kernel tem que dar a mesma KL que a emulação por borda (diferença dentro do ±).

In [ ]:
import math, gguf
from IPython.display import Markdown, display
r = gguf.GGUFReader(f'{G}/Q4_K_M_imat.gguf')
n_q6 = sum(int(t.n_elements) for t in r.tensors if t.name.startswith('blk.') and t.tensor_type.name == 'Q6_K' and t.name.endswith('.weight') and t.n_elements > 1e6)
n_cam = sum(int(t.n_elements) for t in r.tensors if t.name.startswith('blk.') and t.tensor_type.name in ('Q4_K', 'Q6_K') and t.n_elements > 1e6)
EMP = n_q6 / 8 / 2**30                     # GiB a menos no nível 1 com o Q6_K empacotado
KQ = ['Q4_K_M_imat', 'Q5_K_M_imat', 'Q6_K_imat', 'Q8_0']

def pontos():
    return sorted((tam['F16_' + q], kld['F16_' + q][0]) for q in KQ if 'F16_' + q in kld)

def curva(s):
    # KL do k-quant em F16 interpolada (log) no tamanho s (GiB)
    pts = pontos()
    for (s1, k1), (s2, k2) in zip(pts, pts[1:]):
        if s <= s2 or (s2, k2) == pts[-1]:
            if s < pts[0][0]:
                (s1, k1), (s2, k2) = pts[0], pts[1]
            return k1 * math.exp(math.log(k2 / k1) * (s - s1) / (s2 - s1))

def tam_igual(k):
    # tamanho (GiB) do k-quant em F16 com KL k, pela mesma interpolação
    pts = pontos()
    for (s1, k1), (s2, k2) in zip(pts, pts[1:]):
        if k >= k2 or (s2, k2) == pts[-1]:
            if k > pts[0][1]:
                (s1, k1), (s2, k2) = pts[0], pts[1]
            return s1 + (s2 - s1) * math.log(k / k1) / math.log(k2 / k1)

def bits(s, k):
    return (s - tam_igual(k)) * 2**30 * 8 / n_cam

linhas = []
def regua(nome, nivel, s=None):
    if nome not in kld:
        return
    s = tam[nome] if s is None else s
    k = kld[nome][0]
    se = s - EMP if nivel == 1 else s
    linhas.append(f'| {nome} | {nivel} | {s:.3f} | {1000 * k:.2f} ± {1000 * kld[nome][1]:.2f} | {1000 * curva(s):.2f} | {k / curva(s):.2f} | '
                  f'{bits(s, k):+.3f} | {bits(se, k):+.3f} |')

T = [f'# STRATA, sub-células por tensor no kernel (Llama-3.1-8B, llama.cpp CUDA)', '',
     f'GPU: {GPU}. Conferência do CUDA: {"tudo OK" if CUDA_OK else "DIVERGE"}. KL contra o BF16 nos {KLD_CHUNKS} primeiros trechos do {TEXTO} de teste; '
     f'imatrix com {IMATRIX_CHUNKS} trechos do texto de treino. Matrizes das camadas: {n_cam / 1e9:.2f} bilhões de pesos, {n_q6 / 1e9:.2f} bilhões em Q6_K.', '',
     '## KL e tamanho', '', '| modelo | GGUF (GiB) | KL (×1000) | mesmo 1º token |', '|---|---:|---:|---:|']
for m, v in kld.items():
    T.append(f'| {m} | {tam[m]:.3f} | {1000 * v[0]:.2f} ± {1000 * v[1]:.2f} | {v[2]:.2f}% |')
T += ['', '## Régua (curva = k-quants com imatrix em F16)', '',
      '| candidato | nível | RAM (GiB) | KL ×1000 | k-quant F16 no mesmo tamanho | razão | bits/peso a mais | com Q6_K empacotado |',
      '|---|---:|---:|---:|---:|---:|---:|---:|']
for k in (1, 2):
    regua(f'P_N{k}', k)
    regua(f'D_N{k}', k)
    if f'P_N{k}' in tam:
        regua(f'F16_fixo_N{k}', k, s=tam[f'P_N{k}'])
        for por in EMULAR:
            nome = f'F16_{por}_N{k}'
            tab = sum(tabelas.get(f'F16_{por}_N{j}', 0) for j in range(1, k + 1))
            regua(nome, k, s=tam[f'P_N{k}'] + tab / 1024)
T += linhas
T += ['', 'Seção 28 (mesmo 8B, emulação): hoje +0,34 / +0,20 bits/peso a mais (níveis 1 / 2, nível 1 com Q6_K empacotado); sub-células por tensor +0,21 / +0,10.']
if 'D_N1' in kld and 'F16_borda_N1' in kld:
    T += ['', '## Kernel contra a emulação (mesma conta; só o arredondamento dos pesos para F16 muda)', '', '| nível | D (kernel) | emulação por borda | diferença em ± |', '|---:|---:|---:|---:|']
    for k in (1, 2):
        a, b = kld[f'D_N{k}'], kld[f'F16_borda_N{k}']
        T.append(f'| {k} | {1000 * a[0]:.2f} | {1000 * b[0]:.2f} | {(a[0] - b[0]) / math.hypot(a[1], b[1]):+.1f} |')
if vel:
    T += ['', '## Velocidade na GPU (tok/s)', '', '| modelo | geração (tg128) | prompt (pp512) |', '|---|---:|---:|']
    for m, v in vel.items():
        T.append(f'| {m} | {v.get("tg", float("nan")):.1f} | {v.get("pp", float("nan")):.0f} |')
    for a, b in [('D_N1', 'P_N1'), ('D_N2', 'P_N2'), ('D_N1', 'Q5_K_M_imat'), ('D_N2', 'Q6_K_imat')]:
        if a in vel and b in vel:
            T.append(f'| {a} / {b} | {vel[a]["tg"] / vel[b]["tg"]:.2f} | {vel[a]["pp"] / vel[b]["pp"]:.2f} |')

os.makedirs('/content/res', exist_ok=True)
open('/content/res/strata_8b_subcelulas.md', 'w').write('\n'.join(T) + '\n')
json.dump({'texto': TEXTO, 'gpu': GPU, 'cuda_ok': CUDA_OK, 'kld_chunks': KLD_CHUNKS, 'imatrix_chunks': IMATRIX_CHUNKS, 'kld': kld,
           'gguf_gib': tam, 'vel': vel, 'tabelas_mib': tabelas, 'n_q6': n_q6, 'n_camadas': n_cam},
          open('/content/res/strata_8b_subcelulas.json', 'w'), indent=1)
display(Markdown('\n'.join(T)))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/strata_8b_subcelulas.md')
files.download('/content/res/strata_8b_subcelulas.json')